# 🧠 PharMinds Private AI Brain (Cloud Edition)

This notebook runs a private **Large Language Model (LLM)** on Google's cloud GPUs and connects it to your local PharMinds application.

### 🚀 Setup Instructions:
1. Go to **Edit -> Notebook settings** and ensure **Hardware accelerator** is set to **T4 GPU**.
2. Run the cells below.
3. Enter your **Ngrok Authtoken** when prompted (get it for free at [ngrok.com](https://ngrok.com)).
4. Copy the generated `ngrok.io` URL and put it in your PharMinds `.env` file.

In [ ]:
# 1. Install Dependencies
!pip install -q -U transformers bitsandbytes accelerate fastapi uvicorn pyngrok nest-asyncio

In [ ]:
# 2. Configuration & Model Loading
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

print(f"Loading model {model_id} (this may take 5-10 minutes)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Model loaded successfully on Cloud GPU!")

In [ ]:
# 3. Setup FastAPI Server
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import json
import re

app = FastAPI(title="PharMinds Cloud Brain")

@app.post("/v1/chat/completions")
async def chat_completion(request: Request):
    data = await request.json()
    messages = data.get("messages", [])
    
    # Format prompt for Mistral
    prompt = ""
    for msg in messages:
        if msg["role"] == "user":
            prompt += f"[INST] {msg['content']} [/INST]"
        elif msg["role"] == "assistant":
            prompt += f"{msg['content']}"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=True
        )
    
    response_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract only the assistant part (after the last [INST])
    if "[/INST]" in response_text:
        response_text = response_text.split("[/INST]")[-1].strip()
    
    return {
        "choices": [
            {
                "message": {
                    "role": "assistant",
                    "content": response_text
                }
            }
        ]
    }

@app.get("/")
def health():
    return {"status": "PharMinds Brain is Online"}

# 4. Launch Tunnel & Server
print("\n--- ENTER YOUR NGROK AUTHTOKEN BELOW ---")
from google.colab import userdata
token = input("Ngrok Token: ")
ngrok.set_auth_token(token)

public_url = ngrok.connect(8000)
print(f"\n🚀 Your Private Brain is live at: {public_url.public_url}")
print("Copy this URL to your PharMinds .env as VITE_CUSTOM_AI_URL")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)